In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# read your csv
df = pd.read_csv("../data/geocoded_data/merged_data_with_categories_score.csv")

# keep only valid coordinates
df = df[df["ok"] == True]
df = df.dropna(subset=["lat", "lng"])

print(df.shape)
df.head()


C:\Users\shrey\AppData\Local\Temp\ipykernel_36312\3180040610.py:6: DtypeWarning: Columns (3,4,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/geocoded_data/merged_data_with_categories_score.csv")


(413131, 23)


,Opened,Description,Address,Zip Code,Closed Date 1,Closed Date 2,Status,Number,closed,resolution_time_hours,...,address_norm,lat,lng,ok,year,Short Description,Description_norm,category,category_score,category_score_norm
0,2021-02-03 13:30:00,Information on How to Request an Account Adjus...,"4501 Dudley LN NW, Atlanta, 30327",30327,2021-09-13 11:28:00,NaN,Resolved,CS0063095,2021-09-13 11:28:00,4677.982222,...,"4501 Dudley LN NW, Atlanta, 30327",33.877556,-84.388201,True,2021,NaN,information on how to request an account adjus...,Administrative / Account Services,4,0.4
1,2021-03-03 13:30:00,Information on How to Dispute Your Water and S...,"238 peachtree cir, ATLANTA, 30309",30309,2021-07-15 12:54:00,NaN,Resolved,CS0060088,2021-07-15 12:54:00,3215.405833,...,"238 peachtree cir, ATLANTA, 30309",33.795376,-84.386323,True,2021,NaN,information on how to dispute your water and s...,Water & Sewer,9,0.9
2,2021-03-03 13:30:00,Information on How to Dispute Your Water and S...,"3434 Habersham Rd NW, Atlanta, 30305",30305,2022-05-26 09:54:00,NaN,Resolved,CS0026984,2022-05-26 09:54:00,10772.409440,...,"3434 Habersham Rd NW, Atlanta, 30305",33.848637,-84.390632,True,2021,NaN,information on how to dispute your water and s...,Water & Sewer,9,0.9
3,2021-09-03 13:30:00,Street Light Bulb Replacement or Street Light ...,"1160 Veltrie Circle, ATLANTA, 30311",30311,2021-04-14 15:24:00,NaN,Resolved,CS0001758,2021-04-14 15:24:00,865.910278,...,"1160 Veltrie Circle, ATLANTA, 30311",33.723403,-84.475014,True,2021,NaN,street light bulb replacement or street light ...,Road & Infrastructure,8,0.8
4,2021-11-03 13:30:00,Right of Way Maintenance Visibility/Overgrowth...,"2855 elliott cir , ATLANTA, 30305",30305,NaN,2025-03-06 17:47:00,Resolved,CS0056768,2025-06-03 17:47:00,37084.283330,...,"2855 elliott cir, ATLANTA, 30305",33.833415,-84.367372,True,2021,NaN,right of way maintenance visibility/overgrowth...,Road & Infrastructure,8,0.8


In [2]:
# create geometry column
geometry = [Point(xy) for xy in zip(df["lng"], df["lat"])]

gdf_points = gpd.GeoDataFrame(
    df,
    geometry=geometry,
    crs="EPSG:4326"  # WGS84 lat/lon
)

gdf_points.head()


,Opened,Description,Address,Zip Code,Closed Date 1,Closed Date 2,Status,Number,closed,resolution_time_hours,...,lat,lng,ok,year,Short Description,Description_norm,category,category_score,category_score_norm,geometry
0,2021-02-03 13:30:00,Information on How to Request an Account Adjus...,"4501 Dudley LN NW, Atlanta, 30327",30327,2021-09-13 11:28:00,NaN,Resolved,CS0063095,2021-09-13 11:28:00,4677.982222,...,33.877556,-84.388201,True,2021,NaN,information on how to request an account adjus...,Administrative / Account Services,4,0.4,POINT (-84.3882 33.87756)
1,2021-03-03 13:30:00,Information on How to Dispute Your Water and S...,"238 peachtree cir, ATLANTA, 30309",30309,2021-07-15 12:54:00,NaN,Resolved,CS0060088,2021-07-15 12:54:00,3215.405833,...,33.795376,-84.386323,True,2021,NaN,information on how to dispute your water and s...,Water & Sewer,9,0.9,POINT (-84.38632 33.79538)
2,2021-03-03 13:30:00,Information on How to Dispute Your Water and S...,"3434 Habersham Rd NW, Atlanta, 30305",30305,2022-05-26 09:54:00,NaN,Resolved,CS0026984,2022-05-26 09:54:00,10772.409440,...,33.848637,-84.390632,True,2021,NaN,information on how to dispute your water and s...,Water & Sewer,9,0.9,POINT (-84.39063 33.84864)
3,2021-09-03 13:30:00,Street Light Bulb Replacement or Street Light ...,"1160 Veltrie Circle, ATLANTA, 30311",30311,2021-04-14 15:24:00,NaN,Resolved,CS0001758,2021-04-14 15:24:00,865.910278,...,33.723403,-84.475014,True,2021,NaN,street light bulb replacement or street light ...,Road & Infrastructure,8,0.8,POINT (-84.47501 33.7234)
4,2021-11-03 13:30:00,Right of Way Maintenance Visibility/Overgrowth...,"2855 elliott cir , ATLANTA, 30305",30305,NaN,2025-03-06 17:47:00,Resolved,CS0056768,2025-06-03 17:47:00,37084.283330,...,33.833415,-84.367372,True,2021,NaN,right of way maintenance visibility/overgrowth...,Road & Infrastructure,8,0.8,POINT (-84.36737 33.83342)


In [3]:
import os
import requests
import zipfile

url = "https://www2.census.gov/geo/tiger/TIGER2020/TABBLOCK20/tl_2020_13_tabblock20.zip"
zip_path = "tl_2020_13_tabblock20.zip"
out_dir = "tl_2020_13_tabblock20"

# download
if not os.path.exists(zip_path):
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            if chunk:
                f.write(chunk)

# unzip
if not os.path.exists(out_dir):
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)

# read shapefile
shp_path = os.path.join(out_dir, "tl_2020_13_tabblock20.shp")
blocks_ga = gpd.read_file(shp_path)

print(blocks_ga.columns)


Index(['STATEFP20', 'COUNTYFP20', 'TRACTCE20', 'BLOCKCE20', 'GEOID20',
       'GEOIDFQ20', 'NAME20', 'MTFCC20', 'UR20', 'UACE20', 'FUNCSTAT20',
       'ALAND20', 'AWATER20', 'INTPTLAT20', 'INTPTLON20', 'HOUSING20', 'POP20',
       'geometry'],
      dtype='object')


In [4]:
import requests
import zipfile
import os

url_place = "https://www2.census.gov/geo/tiger/TIGER2020/PLACE/tl_2020_13_place.zip"
zip_place = "tl_2020_13_place.zip"
out_place = "tl_2020_13_place"

# 1) Download
if not os.path.exists(zip_place):
    print("Downloading PLACE shapefile...")
    r = requests.get(url_place, stream=True)
    r.raise_for_status()
    with open(zip_place, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            if chunk:
                f.write(chunk)

# 2) Unzip
if not os.path.exists(out_place):
    os.makedirs(out_place, exist_ok=True)
    with zipfile.ZipFile(zip_place, "r") as z:
        z.extractall(out_place)

print("Download complete.")


Download complete.


In [5]:
import geopandas as gpd
import os

place_path = os.path.join(out_place, "tl_2020_13_place.shp")
places = gpd.read_file(place_path)

print(places.head())


  STATEFP PLACEFP   PLACENS    GEOID         NAME          NAMELSAD LSAD  \
0      13   22752  02406378  1322752       Dexter       Dexter town   43   
1      13   24488  02403513  1324488       Dudley       Dudley city   25   
2      13   25300  02403532  1325300  East Dublin  East Dublin city   25   
3      13   52500  02406183  1352500     Montrose     Montrose town   43   
4      13   36696  02403802  1336696       Harlem       Harlem city   25   

  CLASSFP PCICBSA PCINECTA  MTFCC FUNCSTAT     ALAND  AWATER     INTPTLAT  \
0      C1       N        N  G4110        A   1987890   47045  +32.4331950   
1      C1       N        N  G4110        A   8827898   33640  +32.5334829   
2      C1       N        N  G4110        A  10925435  507602  +32.5469249   
3      C1       N        N  G4110        A   4181073   16200  +32.5592295   
4      C1       N        N  G4110        A  16748090   42274  +33.4353393   

       INTPTLON                                           geometry  
0  -083.059

In [6]:
atlanta_city = places[places["NAME"] == "Atlanta"].copy()

# Make sure CRS matches
blocks_ga = blocks_ga.to_crs(atlanta_city.crs)

# Clip blocks using Atlanta boundary
atlanta_blocks = gpd.clip(blocks_ga, atlanta_city)

print("Total Atlanta blocks:", len(atlanta_blocks))
# Georgia State Plane West (feet)
atlanta_blocks = atlanta_blocks.to_crs("EPSG:2240")
gdf_points = gdf_points.to_crs("EPSG:2240")

joined = gpd.sjoin(
    gdf_points,
    atlanta_blocks,
    how="left",
    predicate="within"
)


Total Atlanta blocks: 6575


In [7]:
points_in_city = gpd.sjoin(
    gdf_points,
    atlanta_city.to_crs(gdf_points.crs),
    predicate="within",
    how="inner"
)

print("Points inside Atlanta:", len(points_in_city))


Points inside Atlanta: 407200


In [8]:
atlanta_boundary = atlanta_city.to_crs(gdf_points.crs)

gdf_points_atl = gdf_points[
    gdf_points.within(atlanta_boundary.geometry.iloc[0])
].copy()

print(len(gdf_points_atl))  # should be 437242


407200


In [9]:
joined = gpd.sjoin(
    gdf_points_atl,
    atlanta_blocks,
    how="left",
    predicate="within"
)

print("Unmatched after filtering:",
      joined["GEOID20"].isna().sum())


Unmatched after filtering: 0


In [10]:
# Normalised based on area

atlanta_blocks["area_acres"] = atlanta_blocks.geometry.area / 43560
atlanta_blocks = atlanta_blocks[
    atlanta_blocks["area_acres"] > 0.01
].copy()

#Deep copy of the base data - atlanta_blocks - used for 5 year analysis
atlanta_blocks_base = atlanta_blocks[[
    "GEOID20",
    "geometry",
    "area_acres",
    "POP20"
]].copy()


In [11]:
#Join with our block data
joined_all = gpd.sjoin(
    gdf_points_atl,
    atlanta_blocks[["GEOID20", "geometry"]],
    how="left",
    predicate="within"
)


In [12]:
import numpy as np

gdf_points_atl["response_score"] = np.select(
    [
        gdf_points_atl["resolution_time_hours"] <= 72,
        gdf_points_atl["resolution_time_hours"].between(72, 168),
        gdf_points_atl["resolution_time_hours"] > 168
    ],
    [10, 5, 1],
    default=np.nan
)

In [13]:
all_year_blocks = []

years = sorted(gdf_points_atl["year"].unique())

for yr in years:

    yearly_points = gdf_points_atl[gdf_points_atl["year"] == yr]

    joined_year = gpd.sjoin(
        yearly_points,
        atlanta_blocks_base,
        how="left",
        predicate="within"
    )

    # ----------------------------
    # Complaint counts
    # ----------------------------
    counts = (
        joined_year.groupby("GEOID20")
        .size()
        .reset_index(name="complaint_count")
    )

    blocks_year = atlanta_blocks_base.merge(
        counts,
        on="GEOID20",
        how="left"
    )

    blocks_year["complaint_count"] = blocks_year["complaint_count"].fillna(0)

    # ----------------------------
    # BURDEN SCORE
    # ----------------------------
    blocks_year["burden"] = blocks_year["complaint_count"] / blocks_year["area_acres"]
    blocks_year["log_burden"] = np.log1p(blocks_year["burden"])

    min_val = blocks_year["log_burden"].min()
    max_val = blocks_year["log_burden"].max()

    blocks_year["burden_score"] = (
        (blocks_year["log_burden"] - min_val) /
        (max_val - min_val)
    )

    # ----------------------------
    # RESPONSE SCORE (same method as before)
    # ----------------------------
    block_response = (
        joined_year.groupby("GEOID20")["response_score"]
        .mean()
        .reset_index(name="mean_response_score")
    )

    blocks_year = blocks_year.merge(
        block_response,
        on="GEOID20",
        how="left"
    )

    min_resp = blocks_year["mean_response_score"].min(skipna=True)
    max_resp = blocks_year["mean_response_score"].max(skipna=True)

    blocks_year["response_score_norm"] = (
        (blocks_year["mean_response_score"] - min_resp) /
        (max_resp - min_resp)
    )

    # ----------------------------
    # SEVERITY SCORE
    # ----------------------------
    block_severity = (
        joined_year.groupby("GEOID20")["category_score"]
        .mean()
        .reset_index(name="mean_category_score")
    )

    blocks_year = blocks_year.merge(
        block_severity,
        on="GEOID20",
        how="left"
    )

    min_cat = blocks_year["mean_category_score"].min(skipna=True)
    max_cat = blocks_year["mean_category_score"].max(skipna=True)

    blocks_year["category_score_norm"] = (
        (blocks_year["mean_category_score"] - min_cat) /
        (max_cat - min_cat)
    )

    # ----------------------------
    # COMPOSITE SCORE
    # ----------------------------
    blocks_year["composite_score"] = (
        blocks_year["burden_score"] +
        (1 - blocks_year["response_score_norm"]) +
        blocks_year["category_score_norm"]
    ) / 3

    blocks_year["year"] = yr

    all_year_blocks.append(blocks_year)

In [14]:
blocks_yearly = pd.concat(all_year_blocks, ignore_index=True)
blocks_yearly.head()

,GEOID20,geometry,area_acres,POP20,complaint_count,burden,log_burden,burden_score,mean_response_score,response_score_norm,mean_category_score,category_score_norm,composite_score,year
0,131210072002007,"POLYGON ((2236086.942 1327190.777, 2236120.724...",16.297803,13,10.0,0.613580,0.478455,0.136322,6.3,0.588889,8.2,0.700000,0.415811,2021
1,131210072001015,"POLYGON ((2234870.911 1327348.872, 2235003.084...",24.557998,0,2.0,0.081440,0.078293,0.022307,10.0,1.000000,5.0,0.166667,0.062991,2021
2,131210072001014,"POLYGON ((2234926.087 1327552.891, 2234938.296...",2.711279,0,0.0,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,2021
3,131210072003014,"POLYGON ((2237970.659 1328180.194, 2237993.497...",1.785515,0,0.0,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,2021
4,131210072001013,"POLYGON ((2234226.306 1327159.115, 2234250.087...",23.034531,0,1.0,0.043413,0.042497,0.012108,10.0,1.000000,10.0,1.000000,0.337369,2021


In [15]:
blocks_yearly["threshold"] = blocks_yearly.groupby("year")["composite_score"].transform(
    lambda x: x.quantile(0.9)
)

blocks_yearly["is_worst10"] = blocks_yearly["composite_score"] >= blocks_yearly["threshold"]


In [16]:
blocks_yearly["best_threshold"] = (
    blocks_yearly.groupby("year")["composite_score"]
    .transform(lambda x: x.quantile(0.1))
)

blocks_yearly["is_best10"] = (
    blocks_yearly["composite_score"] <= blocks_yearly["best_threshold"]
)

In [17]:
def classify(row):
    if row["is_worst10"]:
        return "Worst"
    elif row["is_best10"]:
        return "Best"
    else:
        return "Middle"

blocks_yearly["block_category"] = blocks_yearly.apply(classify, axis=1)

In [18]:
blocks_yearly = blocks_yearly.sort_values(["GEOID20","year"])

blocks_yearly["next_year_block_category"] = (
    blocks_yearly.groupby("GEOID20")["block_category"].shift(-1)
)

transition_matrix = pd.crosstab(
    blocks_yearly["block_category"],
    blocks_yearly["next_year_block_category"]
)

print(transition_matrix)

next_year_block_category  Best  Middle  Worst
block_category                               
Best                       386    1411    129
Middle                    1440   17712   1420
Worst                      124    1401    401


In [26]:
blocks_yearly

,GEOID20,geometry,area_acres,POP20,complaint_count,burden,log_burden,burden_score,mean_response_score,response_score_norm,mean_category_score,category_score_norm,composite_score,year,threshold,is_worst10,best_threshold,is_best10,block_category,next_year_block_category
4704,130890201001000,"POLYGON ((2244128.741 1383758.744, 2244205.379...",8.885744,216,4.0,0.450159,0.371673,0.105897,4.250000,0.361111,7.250000,0.541667,0.428818,2021,0.508748,False,0.297352,False,Middle,Middle
10810,130890201001000,"POLYGON ((2244128.741 1383758.744, 2244205.379...",8.885744,216,0.0,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,2022,0.532196,False,0.248792,False,Middle,Middle
16916,130890201001000,"POLYGON ((2244128.741 1383758.744, 2244205.379...",8.885744,216,11.0,1.237938,0.805555,0.225367,7.454545,0.717172,8.090909,0.681818,0.396671,2023,0.470565,False,0.265799,False,Middle,Middle
23022,130890201001000,"POLYGON ((2244128.741 1383758.744, 2244205.379...",8.885744,216,9.0,1.012858,0.699556,0.218234,5.222222,0.469136,7.222222,0.537037,0.428712,2024,0.508668,False,0.286140,False,Middle,Middle
29128,130890201001000,"POLYGON ((2244128.741 1383758.744, 2244205.379...",8.885744,216,20.0,2.250796,1.178900,0.287647,7.950000,0.772222,8.150000,0.691667,0.402364,2025,0.489256,False,0.266819,False,Middle,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34,131219800001011,"POLYGON ((2217083.564 1330135.599, 2217051.81 ...",0.955174,0,0.0,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,2021,0.508748,False,0.297352,False,Middle,Middle
6140,131219800001011,"POLYGON ((2217083.564 1330135.599, 2217051.81 ...",0.955174,0,0.0,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,2022,0.532196,False,0.248792,False,Middle,Middle
12246,131219800001011,"POLYGON ((2217083.564 1330135.599, 2217051.81 ...",0.955174,0,0.0,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,2023,0.470565,False,0.265799,False,Middle,Middle
18352,131219800001011,"POLYGON ((2217083.564 1330135.599, 2217051.81 ...",0.955174,0,0.0,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,2024,0.508668,False,0.286140,False,Middle,Middle


In [19]:
worst_counts = (
    blocks_yearly[blocks_yearly["is_worst10"]]
    .groupby("GEOID20")["year"]
    .nunique()
)

persistent_worst = worst_counts[worst_counts >= 3]

print("Persitent worst:", len(persistent_worst))


best_counts = (
    blocks_yearly[blocks_yearly["is_best10"]]
    .groupby("GEOID20")["year"]
    .nunique()
)

persistent_best = best_counts[best_counts >= 3]

print("Persistent best:", len(persistent_best))


improved_blocks = blocks_yearly[
    (blocks_yearly["block_category"] == "Worst") &
    (blocks_yearly["next_year_block_category"] == "Best")
]["GEOID20"].unique()

print("Improved blocks:", len(improved_blocks))

deteriorated_blocks = blocks_yearly[
    (blocks_yearly["block_category"] == "Best") &
    (blocks_yearly["next_year_block_category"] == "Worst")
]["GEOID20"].unique()

print("Deteriorated blocks:", len(deteriorated_blocks))

Persitent worst: 152
Persistent best: 146
Improved blocks: 123
Deteriorated blocks: 128


In [23]:
worst_points = joined_all[
    joined_all["GEOID20"].isin(persistent_worst.index)
]

worst_issue_counts = worst_points["category"].value_counts()

best_points = joined_all[
    joined_all["GEOID20"].isin(persistent_best.index)
]

best_issue_counts = best_points["category"].value_counts()

improved_points = joined_all[
    joined_all["GEOID20"].isin(improved_blocks)
]

improved_issue_counts = improved_points["category"].value_counts()

deteriorated_points = joined_all[
    joined_all["GEOID20"].isin(deteriorated_blocks)
]

deteriorated_issue_counts = deteriorated_points["category"].value_counts()

In [24]:
worst_issue_pct = worst_points["category"].value_counts(normalize=True) * 100
best_issue_pct = best_points["category"].value_counts(normalize=True) * 100
improved_issue_pct = improved_points["category"].value_counts(normalize=True) * 100
deteriorated_issue_pct = deteriorated_points["category"].value_counts(normalize=True) * 100

In [25]:
issue_comparison = pd.concat(
    [
        worst_issue_pct,
        best_issue_pct,
        improved_issue_pct,
        deteriorated_issue_pct
    ],
    axis=1
)

issue_comparison.columns = [
    "Persistent Worst %",
    "Persistent Best %",
    "Improved %",
    "Deteriorated %"
]

issue_comparison = issue_comparison.fillna(0)

print(issue_comparison)

                                   Persistent Worst %  Persistent Best %  \
category                                                                   
Waste Management                            48.617966          28.475621   
Code Enforcement                            13.795656          22.711619   
Water & Sewer                               12.179171          15.225666   
Road & Infrastructure                       11.377098           5.528367   
Public Safety                                8.970879           7.775965   
Licensing & Permits                          2.702369          14.990031   
Administrative / Account Services            2.356861           5.292732   

                                   Improved %  Deteriorated %  
category                                                       
Waste Management                    39.410555       37.384818  
Code Enforcement                    14.633310       15.971918  
Water & Sewer                       16.004112       11.8911